## Train an emoji ReFT intervention with EasySteer


Adapted from the original [pyreft tutorial](https://github.com/stanfordnlp/pyreft). This notebook uses the vendored implementation through EasySteer’s shared training helpers.


Install this checkout and its vllm-steer dependency using the project installation guide, then select that Python environment as the notebook kernel. The example uses Qwen2.5-1.5B-Instruct and requires a GPU for the default device.


In [ ]:
import os
from pathlib import Path

from easysteer.reft.train import (
    EMOJI_EXAMPLES, generate_reft, load_reft, train_reft,
)


### Step 1: configure the model and device

The shared helper’s default prompt template is the Qwen chat format. Set `REFT_MODEL_PATH` to a local copy of Qwen2.5-1.5B-Instruct if needed. For another model family, supply a matching `prompt_template` to both training and generation. Select the physical GPU before starting this kernel.


In [ ]:
model_name_or_path = os.environ.get("REFT_MODEL_PATH", "Qwen/Qwen2.5-1.5B-Instruct")
device = "cuda"


### Step 2: choose the intervention

A rank-4 LoReFT intervention targets layer 8 at the last prompt token. These settings are passed through the shared helper, which constructs the local `ReftConfig` and the appropriate Transformers trainer.


In [ ]:
layer = 8
rank = 4
save_dir = Path("results/emoji_loreft")


### Step 3: choose demonstrations

The shared emoji demonstration set contains instruction/response pairs. A model trained on so few examples can memorize them and may not generalize.


In [ ]:
training_examples = EMOJI_EXAMPLES
training_examples[:2]


### Step 4: train and save locally

The helper adapts to the installed Transformers trainer interface and saves the intervention checkpoint in `save_dir`. This cell does not publish to Hugging Face.


In [ ]:
reft_model, tokenizer = train_reft(
    model_path=model_name_or_path,
    examples=training_examples,
    intervention="loreft",
    layer=layer,
    low_rank_dimension=rank,
    device=device,
    num_train_epochs=100.0,
    per_device_train_batch_size=10,
    learning_rate=4e-3,
    logging_steps=40,
    output_dir=str(save_dir / "training"),
    save_dir=str(save_dir),
)


### Step 5: generate with the trained intervention

`generate_reft` applies the intervention at the last prompt position, matching training.


In [ ]:
instruction = "Which dog breed do people think is cuter, poodle or doodle?"
print(generate_reft(reft_model, tokenizer, instruction, device=device))


### Step 6: prepare the checkpoint for vllm-steer

The current inference API accepts a canonical payload through `VectorSpec(data=...)`. A LoReFT checkpoint uses `algorithm="loreft"`; a BiasIntervention checkpoint instead uses `algorithm="direct"`. The adapter preserves the checkpoint’s layer.


In [ ]:
import easysteer.vectors as vec

payload = vec.from_pyreft(str(save_dir))
print(payload.kind)  # "reft" for this LoReFT example


### Step 7: reload the saved intervention

Reload onto the same base model. Release the training instance first; when moving on to a separate vLLM engine, restart the kernel or otherwise release its model memory.


In [ ]:
import gc
import torch

del reft_model
gc.collect()
torch.cuda.empty_cache()
reft_model, tokenizer = load_reft(model_name_or_path, str(save_dir), device=device)
print(generate_reft(reft_model, tokenizer, "Who are you?", device=device))


### Next: vllm-steer inference or the web demo

See the repository’s `docs/user-guide/reft-training.md` for a complete `SteeringSpec` / `VectorSpec(data=vec.from_pyreft(...))` inference example, and `frontend/README.md` for training through the web UI. They use the same shared training helper as this notebook.
